# CS570 Team 14 — GCN Modifiability Prediction
## Colab Setup & Training Notebook

---

## ⚠️ BEFORE YOU RUN ANYTHING — Read This First

There are a few things you need to do manually before the code will work.
Do these **once** at the start of every new Colab session.

---

### Manual Step 1 — Enable GPU

By default Colab gives you a CPU, which is too slow for training. You need to switch to GPU:

1. In the top menu, click **Runtime**
2. Click **Change runtime type**
3. Under **Hardware accelerator**, select **T4 GPU**
4. Click **Save**
5. Colab will restart — that's normal

How to verify it worked: after the restart, the top-right of the page should show a green chip that says **T4**. If it says **CPU**, repeat the steps above.

---

### Manual Step 2 — Google Account

You need to be logged into a Google account. Colab will ask you to authorize access to Google Drive in Step 2 of the notebook. When the pop-up appears:

1. Click the link it gives you
2. Choose your Google account
3. Click **Allow**
4. Copy the authorization code and paste it back into the Colab input box
5. Press Enter

You will see: `Mounted at /content/drive` — that means it worked.

---

### Manual Step 3 — Rico Dataset (one-time only)

The notebook will try to download the Rico dataset automatically. **If the automatic download fails** (the Rico website sometimes goes down), follow these steps:

1. On your laptop, go to: **http://interactionmining.org/rico**
2. Look for the download section — find **`ui_layout_vectors.zip`** (this is the ~3 GB file with the UI hierarchy JSONs)
3. Download it to your laptop
4. Go to **https://drive.google.com**
5. Navigate into: `My Drive > cs570-project` (create this folder if it doesn't exist yet)
6. Upload `ui_layout_vectors.zip` into that folder
7. Come back to this notebook and run the **Manual Upload Fallback** cell in Section 6

You only need to do this **once ever**. After the `.pt` processed files are on Drive, you never need the raw JSONs again.

---

### What lives where

| Data | Location | Persists after session? |
|------|----------|--------------------------|
| Rico raw JSONs (3 GB) | `/content/rico_raw/` (local) | ❌ No — re-downloaded each session but only needed once |
| Processed `.pt` graphs | Google Drive | ✅ Yes |
| Model checkpoints | Google Drive | ✅ Yes |
| Embedding cache | Google Drive | ✅ Yes |
| This repo | `/content/cs570-aiml/` (local) | ❌ No — re-cloned each session (fast) |

**The key insight:** you only need to download and preprocess Rico once. After that, each new Colab session just loads the `.pt` files from Drive and trains.

---

### How to run the notebook

Run cells **top to bottom, one at a time**. Each cell has a ▶ button on the left. Wait for one cell to finish (the `[*]` next to it becomes a number like `[3]`) before running the next one.

If a cell throws an error, read the last line of the error — it usually tells you exactly what's wrong. The most common issues are covered in the **Troubleshooting** section at the bottom.

---
## Section 1 — Check GPU

Run this first to confirm you have a GPU. If it says `Failed to initialize NVML` or shows no GPU, go back and do Manual Step 1.

In [ ]:
!nvidia-smi

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU ready: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)} GB")
else:
    print("❌ No GPU found. Go to Runtime > Change runtime type > T4 GPU, then re-run.")

---
## Section 2 — Mount Google Drive

This will pop up an authorization prompt. Follow Manual Step 2 above.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
if os.path.exists('/content/drive/MyDrive'):
    print("✅ Drive mounted successfully")
else:
    print("❌ Drive mount failed — try running this cell again")

---
## Section 3 — Clone Repo from GitHub

In [ ]:
import os

REPO_URL = "https://github.com/nadiarvi/cs570-aiml"
REPO_DIR = "/content/cs570-aiml"
BRANCH   = "colab"

if os.path.exists(REPO_DIR):
    print("Repo already cloned — pulling latest changes...")
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    print("Cloning repo...")
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"\n✅ Working directory: {os.getcwd()}")

In [ ]:
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("✅ Repo added to Python path")

---
## Section 4 — Install Dependencies

Torch, numpy, and pandas are pre-installed in Colab. This just installs the missing ones.
Takes about 1–2 minutes.

In [ ]:
!pip install -q sentence-transformers scikit-learn seaborn tqdm
print("✅ Dependencies installed")

---
## Section 5 — Set Up Paths

- **Drive** (`DRIVE_ROOT`): stores processed graphs, checkpoints, embedding cache — persists between sessions
- **Local** (`RICO_RAW_DIR`): stores raw Rico JSONs temporarily — fast to work with, gets wiped when session ends (that's fine)

In [ ]:
import os

# ── Persistent (Google Drive) ──────────────────────────────────────────────
DRIVE_ROOT     = "/content/drive/MyDrive/cs570-project"
PROCESSED_DIR  = f"{DRIVE_ROOT}/data/processed"
SPLITS_DIR     = f"{DRIVE_ROOT}/data/splits"
GOLD_DIR       = f"{DRIVE_ROOT}/data/gold"
CHECKPOINTS    = f"{DRIVE_ROOT}/results/checkpoints"
RESULTS_DIR    = f"{DRIVE_ROOT}/results"
EMBEDDING_CACHE = f"{DRIVE_ROOT}/embedding_cache.json"

# ── Ephemeral (local Colab storage — fast, wiped on session end) ───────────
RICO_RAW_DIR   = "/content/rico_raw"   # raw JSONs live here temporarily
RICO_ZIP_LOCAL = "/content/ui_layout_vectors.zip"

# Create all Drive folders
for path in [PROCESSED_DIR, SPLITS_DIR, GOLD_DIR, CHECKPOINTS,
             f"{RESULTS_DIR}/logs", f"{RESULTS_DIR}/figures"]:
    os.makedirs(path, exist_ok=True)

os.makedirs(RICO_RAW_DIR, exist_ok=True)

print("Drive folders:")
print(f"  processed graphs → {PROCESSED_DIR}")
print(f"  checkpoints      → {CHECKPOINTS}")
print(f"  embedding cache  → {EMBEDDING_CACHE}")
print(f"\nLocal (temporary):")
print(f"  rico raw JSONs   → {RICO_RAW_DIR}")
print("\n✅ Paths configured")

---
## Section 6 — Get Rico Dataset

**Skip this entire section if you have already preprocessed the data in a previous session.**
Run the check cell first to see if you need to re-download.

In [ ]:
# Check if processed graphs already exist on Drive (from a previous session)
import glob

existing_pt = glob.glob(f"{PROCESSED_DIR}/**/*.pt", recursive=True)
if existing_pt:
    print(f"✅ Found {len(existing_pt)} processed .pt files on Drive.")
    print("   You can skip Section 6 and go straight to Section 7 (training).")
else:
    print("⚠️  No processed graphs found. You need to download + preprocess Rico.")
    print("   Continue running the cells below.")

In [ ]:
# ── Automatic download attempt ─────────────────────────────────────────────
# This tries the official Rico URL. If it fails (site is down or slow),
# skip to the manual fallback cell below.

RICO_ZIP_ON_DRIVE = f"{DRIVE_ROOT}/ui_layout_vectors.zip"

# If already on Drive (from a previous manual upload), copy to local
if os.path.exists(RICO_ZIP_ON_DRIVE):
    print("Found zip on Drive — copying to local storage (faster to unzip locally)...")
    !cp {RICO_ZIP_ON_DRIVE} {RICO_ZIP_LOCAL}
    print("✅ Copied")

elif not os.path.exists(RICO_ZIP_LOCAL):
    print("Downloading Rico dataset (~3 GB) — this takes 2–5 minutes...")
    !wget -q --show-progress \
        "http://interactionmining.org/rico/static/raw-data/ui_layout_vectors.zip" \
        -O {RICO_ZIP_LOCAL}

    if os.path.exists(RICO_ZIP_LOCAL) and os.path.getsize(RICO_ZIP_LOCAL) > 1e8:
        print("✅ Download complete")
        # Save a copy to Drive so you don't need to re-download if you need raw JSONs again
        print("Saving a copy to Drive for future use...")
        !cp {RICO_ZIP_LOCAL} {RICO_ZIP_ON_DRIVE}
    else:
        print("❌ Download failed or file is too small.")
        print("   Follow the manual upload instructions in the cell below.")
else:
    print("✅ Zip already in local storage")

In [ ]:
# ── Manual Upload Fallback ─────────────────────────────────────────────────
# ONLY run this cell if the automatic download above failed.
#
# What to do:
#   1. On your laptop, go to: http://interactionmining.org/rico
#   2. Download "ui_layout_vectors.zip" (~3 GB)
#   3. Go to https://drive.google.com
#   4. Navigate to: My Drive > cs570-project
#   5. Upload ui_layout_vectors.zip into that folder
#   6. Wait for the upload to finish (progress bar in Drive bottom-right)
#   7. Then run this cell

RICO_ZIP_ON_DRIVE = f"{DRIVE_ROOT}/ui_layout_vectors.zip"

if os.path.exists(RICO_ZIP_ON_DRIVE):
    size_gb = os.path.getsize(RICO_ZIP_ON_DRIVE) / 1e9
    print(f"✅ Found zip on Drive ({size_gb:.1f} GB) — copying to local storage...")
    !cp {RICO_ZIP_ON_DRIVE} {RICO_ZIP_LOCAL}
    print("✅ Ready to unzip")
else:
    print(f"❌ File not found at: {RICO_ZIP_ON_DRIVE}")
    print("   Make sure you uploaded it to the right folder in Drive.")

In [ ]:
# ── Unzip to local storage ─────────────────────────────────────────────────
# Unzipping to local /content/ is much faster than unzipping directly to Drive.
# Takes ~5–10 minutes for 66K files.

existing_jsons = glob.glob(f"{RICO_RAW_DIR}/**/*.json", recursive=True)

if len(existing_jsons) > 1000:
    print(f"✅ Raw JSONs already unzipped ({len(existing_jsons)} files) — skipping")
elif os.path.exists(RICO_ZIP_LOCAL):
    print(f"Unzipping {RICO_ZIP_LOCAL} → {RICO_RAW_DIR}")
    print("This takes ~5–10 minutes. The cell is done when you see a checkmark.")
    !unzip -q {RICO_ZIP_LOCAL} -d {RICO_RAW_DIR}
    unzipped = glob.glob(f"{RICO_RAW_DIR}/**/*.json", recursive=True)
    print(f"✅ Unzipped {len(unzipped)} JSON files")
else:
    print("❌ No zip file found. Run the download or manual upload cell first.")

In [ ]:
# Verify Rico data is ready
json_files = glob.glob(f"{RICO_RAW_DIR}/**/*.json", recursive=True)
print(f"Rico JSON files found: {len(json_files)}")

if len(json_files) > 10000:
    print("✅ Looks good — ready to preprocess")
    print(f"   Sample path: {json_files[0]}")
elif len(json_files) > 0:
    print(f"⚠️  Only {len(json_files)} files found. Expected ~66,000. Unzip may be incomplete.")
else:
    print("❌ No JSON files found. Something went wrong with the download or unzip.")

---
## Section 7 — Preprocess

Converts raw Rico JSONs → processed graph `.pt` files saved to Drive.

**This is the slowest step** because it runs MiniLM text embeddings on every node's text.
The embedding cache (`embedding_cache.json`) saves results so repeated runs are fast.

**Strategy:**
1. Run the 500-screen sanity check first (~3 minutes) — confirms the pipeline works
2. Then run 10,000 screens (~45–90 minutes) — enough to train a good model
3. Full 66K only if you need it later

**Skip this section** if you already have `.pt` files on Drive from a previous session.

In [ ]:
# ── Step 7a: Sanity check — 500 screens ───────────────────────────────────
# Run this first to make sure the pipeline works end-to-end before committing
# to the full 10K run.

!python src/data/preprocess.py \
    --rico_dir             {RICO_RAW_DIR} \
    --out_dir              {PROCESSED_DIR} \
    --split_path           {SPLITS_DIR}/split_seed42.json \
    --label_mode           contextual \
    --embedding_cache_path {EMBEDDING_CACHE} \
    --workers              1 \
    --max_screens          500

In [ ]:
# Verify sanity check produced files
train_pts = glob.glob(f"{PROCESSED_DIR}/train/**/*.pt", recursive=True)
val_pts   = glob.glob(f"{PROCESSED_DIR}/val/**/*.pt",   recursive=True)
print(f"Train graphs: {len(train_pts)}")
print(f"Val graphs:   {len(val_pts)}")

if train_pts:
    # Quick sanity: load one graph and print its shape
    import torch
    sample = torch.load(train_pts[0], weights_only=False)
    print(f"\nSample graph from: {train_pts[0]}")
    print(f"  Nodes:          {sample['num_nodes']}")
    print(f"  Feature dim:    {sample['x'].shape[1]}")
    print(f"  Edge count:     {sample['edge_index'].shape[1]}")
    print(f"  Label counts:   { {int(k): int(v) for k, v in zip(*sample['y'].unique(return_counts=True))} }")
    print("\n✅ Pipeline working correctly")
else:
    print("❌ No .pt files found — check the error output from the preprocess cell above")

In [ ]:
# ── Step 7b: Full preprocessing — 10K screens, contextual labels ──────────
# Already-processed files are skipped automatically, so re-running is safe.
# Expect ~45–90 minutes. The embedding cache speeds up repeated runs.

!python src/data/preprocess.py \
    --rico_dir             {RICO_RAW_DIR} \
    --out_dir              {PROCESSED_DIR} \
    --split_path           {SPLITS_DIR}/split_seed42.json \
    --label_mode           contextual \
    --embedding_cache_path {EMBEDDING_CACHE} \
    --workers              1 \
    --max_screens          10000

In [ ]:
# ── Step 7c: Full preprocessing — local_only labels ───────────────────────
# Needed for the circularity safeguard experiment (local_only ablation configs).
# Run after 7b finishes.

!python src/data/preprocess.py \
    --rico_dir             {RICO_RAW_DIR} \
    --out_dir              {PROCESSED_DIR} \
    --split_path           {SPLITS_DIR}/split_seed42.json \
    --label_mode           local_only \
    --embedding_cache_path {EMBEDDING_CACHE} \
    --workers              1 \
    --max_screens          10000

In [ ]:
# Final verification before training
for mode in ["contextual", "local_only"]:
    train_pts = glob.glob(f"{PROCESSED_DIR}/train/{mode}/**/*.pt", recursive=True)
    val_pts   = glob.glob(f"{PROCESSED_DIR}/val/{mode}/**/*.pt",   recursive=True)
    print(f"{mode:12s}  train={len(train_pts):5d}  val={len(val_pts):5d}")

---
## Section 8 — Train a Single Model

Trains GCN on heuristic contextual labels. Early-stops on heuristic val Macro-F1.
Checkpoint is saved to Drive so you can resume in a future session.

With 10K screens and 100 epochs, expect **30–60 minutes**.

In [ ]:
import json
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

from src.train import train

with open(f"{REPO_DIR}/experiments/configs/colab_gcn_2l_all_contextual.json") as f:
    config = json.load(f)

# Override with actual runtime paths
config["processed_dir"] = PROCESSED_DIR
config["save_dir"]       = f"{CHECKPOINTS}/gcn_2l_all_contextual"
config["save_root"]      = CHECKPOINTS

metadata = train(config)
print(f"\n✅ Training complete")
print(f"   Best val Macro-F1 : {metadata['best_val_macro_f1']:.4f}")
print(f"   Best epoch         : {metadata['best_epoch']}")
print(f"   Checkpoint saved to: {metadata['checkpoint_path']}")

---
## Section 9 — Gold Labels

⚠️ **Manual step required before running this section.**

You need to create `gold_test_labels.csv` and upload it to Drive before evaluation.
This file contains the hand-labeled annotations for ~50 Rico screens.

**What to do:**
1. Annotate ~50 screens from apps that are NOT in the heuristic train/val split
   - To find which apps are in the gold holdout, look at `data/splits/split_seed42.json` on Drive after preprocessing — specifically the `gold_app_ids` list
   - If you haven't decided which apps to annotate yet, pick any 50 apps NOT in `train_app_ids` or `val_app_ids`
2. Create a CSV file with these columns: `screen_id, node_id, label, annotator_id, notes`
   - `label` must be one of: `canonical`, `translatable`, `open`
   - `node_id` must match the `node_id` field in the flattened hierarchy (run the inspect cell below to check)
3. Save the file as `gold_test_labels.csv`
4. Upload it to `My Drive > cs570-project > data > gold` in Google Drive
5. Then run the evaluation cells below

**The cell below lets you inspect node IDs for a specific screen to help with annotation.**

In [ ]:
# Inspect node IDs for a specific screen to help with annotation
# Change SCREEN_PATH to the .pt file you want to annotate

import torch, json

# List a few available screens to pick from
sample_pts = glob.glob(f"{PROCESSED_DIR}/train/contextual/**/*.pt", recursive=True)[:5]
print("Sample .pt files available:")
for p in sample_pts:
    print(" ", p)

print("\nChange SCREEN_PATH below to inspect a specific screen's nodes:")
SCREEN_PATH = sample_pts[0] if sample_pts else None

if SCREEN_PATH:
    g = torch.load(SCREEN_PATH, weights_only=False)
    print(f"\nScreen: {g['screen_id']}  App: {g['app_id']}  Nodes: {g['num_nodes']}")
    print(f"{'node_id':>8}  {'label':>12}  {'depth':>5}")
    print("-" * 30)
    label_names = {0: 'canonical', 1: 'translatable', 2: 'open', -1: 'excluded'}
    for i, (nid, lbl) in enumerate(zip(g['node_ids'], g['y'].tolist())):
        print(f"{nid:>8}  {label_names.get(lbl, str(lbl)):>12}")
        if i >= 20:
            print(f"  ... ({g['num_nodes'] - 21} more nodes)")
            break

---
## Section 10 — Run Full Ablation Matrix

Trains all 9 experiment configs and evaluates on gold labels.
Results are saved to `ablation_results.csv` on Drive.

**Requires:** gold labels uploaded to Drive (Section 9).

This will take several hours — one run per config × 9 configs.

In [ ]:
from src.ablation import run_ablation

with open(f"{REPO_DIR}/experiments/configs/colab_ablation_base.json") as f:
    base_config = json.load(f)

base_config["processed_dir"]    = PROCESSED_DIR
base_config["gold_labels_path"] = f"{GOLD_DIR}/gold_test_labels.csv"
base_config["save_root"]        = CHECKPOINTS

out_csv = f"{RESULTS_DIR}/ablation_results.csv"
run_ablation(base_config, out_csv=out_csv)
print(f"\n✅ Ablation complete — results at {out_csv}")

In [ ]:
# View results table
import pandas as pd

results = pd.read_csv(out_csv)
cols = ["name", "model", "label_mode", "heuristic_val_macro_f1", "gold_macro_f1", "gold_accuracy"]
print(results[cols].sort_values("gold_macro_f1", ascending=False).to_string(index=False))

---
## Section 11 — Push Results to GitHub

In [ ]:
# Copy results from Drive into the repo, then push
!cp {out_csv} {REPO_DIR}/results/ablation_results.csv

!git -C {REPO_DIR} config user.email "nadia.arvi@gmail.com"
!git -C {REPO_DIR} config user.name "nadiarvi"
!git -C {REPO_DIR} add results/ablation_results.csv
!git -C {REPO_DIR} commit -m "Add ablation results from Colab run"
!git -C {REPO_DIR} push origin colab

---
## Utilities

In [ ]:
# Check GPU memory usage (run this while training is happening in another cell)
!nvidia-smi --query-gpu=memory.used,memory.free,utilization.gpu --format=csv

In [ ]:
# Check how much Drive space you're using
!du -sh {DRIVE_ROOT}/*

In [ ]:
# Check how much time is left in your Colab session
# (Colab free tier disconnects after ~12 hours, Pro after ~24 hours)
!cat /proc/uptime | awk '{print "Session uptime:", int($1/3600), "hours", int(($1%3600)/60), "minutes"}'

---
## Troubleshooting

**`FileNotFoundError: No training graphs found in .../processed/train/contextual/**/*.pt`**
> Preprocessing hasn't run yet. Go to Section 7 and run those cells first.

**`ModuleNotFoundError: No module named 'src'`**
> The repo isn't on the Python path. Re-run the sys.path cell in Section 3.

**`ModuleNotFoundError: No module named 'sentence_transformers'`**
> Re-run Section 4 (pip install cell).

**`RuntimeError: CUDA out of memory`**
> Reduce `batch_size` in the config (try 16 or 8), then re-run the training cell.

**Drive mount shows an error or hangs**
> Run `drive.flush_and_unmount()` then re-run the mount cell. If it still hangs, go to Runtime > Restart runtime and start from Section 1.

**Colab session disconnected mid-training**
> The checkpoint is saved to Drive after every improvement. Re-run Sections 1–4 (setup), then re-run the training cell — it will resume from where it left off if you load the checkpoint. Currently training restarts from scratch; if this is a problem let me know and I can add checkpoint resuming.

**`wget` download is very slow or fails**
> The Rico server can be unreliable. Try the manual upload fallback in Section 6.

**Preprocessing is stuck on the same screen for a long time**
> The MiniLM model is being downloaded on the first run (~90 MB). It'll be fast after that.

**`git push` asks for a password**
> GitHub no longer accepts passwords. Use a Personal Access Token instead:
> 1. Go to https://github.com/settings/tokens
> 2. Click `Generate new token (classic)`
> 3. Check the `repo` scope
> 4. Copy the token
> 5. When git asks for password, paste the token instead